# 01 - Data Exploration: Hassaniya Arabic Speech Dataset

**Project:** Hassaniya Arabic Text-To-Speech System Using Transfer Learning  
**Author:** Mohamed Salem Ebnou Echvagha Oubeid (C34613)  
**Module:** NLP Dialects — Master M1 AI  
**Date:** June 2026

---

## Objective

In this notebook, we explore our Hassaniya Arabic speech dataset to understand:
- How many samples we have
- The distribution of text lengths
- Audio properties (duration, sample rate, waveform shapes)
- Character frequency in Hassaniya text

This exploration is essential before building any TTS system — we need to know
what our data looks like to make informed preprocessing decisions.

## 1. Setup & Imports

In [ ]:
# Install dependencies (run in Colab)
# !pip install pandas pyarrow librosa soundfile matplotlib seaborn numpy tqdm

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import io
import struct
from collections import Counter

# Audio libraries
try:
    import librosa
    import librosa.display
    import soundfile as sf
    AUDIO_AVAILABLE = True
    print('Audio libraries loaded successfully')
except ImportError:
    AUDIO_AVAILABLE = False
    print('Audio libraries not available — text analysis only')

try:
    from IPython.display import Audio, display
    IPYTHON_AVAILABLE = True
except ImportError:
    IPYTHON_AVAILABLE = False

# Plot settings
plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')

print(f'Setup complete')

## 2. Load the Dataset

Our dataset is stored as a **Parquet file** — a columnar format that efficiently
stores both text and binary data (audio bytes). Each row contains:
- `audio`: a dictionary with `bytes` (raw audio data) and `path`
- `transcription`: the Hassaniya Arabic text

In [ ]:
# Load the dataset
df = pd.read_parquet('../audios_dataset.parquet')

print(f'Dataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print(f'\nColumn types:')
print(df.dtypes)
print(f'\nTotal samples: {len(df)}')

In [ ]:
# Examine the structure of the audio column
sample = df['audio'].iloc[0]
print(f'Audio column type: {type(sample)}')
if isinstance(sample, dict):
    print(f'Audio dict keys: {list(sample.keys())}')
    audio_bytes = sample.get('bytes', b'')
    print(f'Audio bytes length: {len(audio_bytes)} bytes')
    print(f'Audio path: {sample.get("path", "None")}')

## 3. Text Analysis

Let's analyze the transcriptions to understand the linguistic characteristics
of our Hassaniya dataset.

In [ ]:
# Display sample transcriptions
print('=== Sample Hassaniya Transcriptions ===')
print()
for i in range(min(10, len(df))):
    print(f'  [{i:3d}] {df["transcription"].iloc[i]}')

In [ ]:
# Text length statistics
df['text_length'] = df['transcription'].str.len()
df['word_count'] = df['transcription'].str.split().str.len()

print('=== Text Length Statistics ===')
print(f'Mean character length: {df["text_length"].mean():.1f}')
print(f'Median character length: {df["text_length"].median():.1f}')
print(f'Min character length: {df["text_length"].min()}')
print(f'Max character length: {df["text_length"].max()}')
print(f'Std deviation: {df["text_length"].std():.1f}')
print()
print('=== Word Count Statistics ===')
print(f'Mean words per sample: {df["word_count"].mean():.1f}')
print(f'Max words per sample: {df["word_count"].max()}')
print(f'Min words per sample: {df["word_count"].min()}')

In [ ]:
# Visualize text length distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Character length distribution
axes[0].hist(df['text_length'], bins=30, color='#2196F3', edgecolor='white', alpha=0.8)
axes[0].set_xlabel('Character Length')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Transcription Lengths (Characters)')
axes[0].axvline(df['text_length'].mean(), color='red', linestyle='--', label=f'Mean: {df["text_length"].mean():.0f}')
axes[0].legend()

# Word count distribution
axes[1].hist(df['word_count'], bins=20, color='#4CAF50', edgecolor='white', alpha=0.8)
axes[1].set_xlabel('Word Count')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Word Counts per Sample')
axes[1].axvline(df['word_count'].mean(), color='red', linestyle='--', label=f'Mean: {df["word_count"].mean():.1f}')
axes[1].legend()

plt.tight_layout()
plt.savefig('../results/figures/text_length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved to results/figures/text_length_distribution.png')

In [ ]:
# Character frequency analysis
all_text = ' '.join(df['transcription'].tolist())
char_counts = Counter(ch for ch in all_text if ch.strip())

# Top 30 most frequent characters
top_chars = char_counts.most_common(30)
chars, counts = zip(*top_chars)

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(range(len(chars)), counts, color='#FF9800', edgecolor='white')
ax.set_xticks(range(len(chars)))
ax.set_xticklabels(chars, fontsize=14)
ax.set_xlabel('Character')
ax.set_ylabel('Frequency')
ax.set_title('Top 30 Most Frequent Characters in Hassaniya Transcriptions')

plt.tight_layout()
plt.savefig('../results/figures/character_frequency.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nTotal unique characters: {len(char_counts)}')
print(f'Total characters (excluding spaces): {sum(char_counts.values())}')

## 4. Audio Analysis

Now let's examine the audio properties of our dataset.
Each audio sample is stored as raw bytes — we decode them to analyze
waveforms, durations, and spectral properties.

In [ ]:
def decode_audio(audio_dict, target_sr=22050):
    """Decode audio bytes from the dataset into a waveform array."""
    audio_bytes = audio_dict.get('bytes', b'')
    if not audio_bytes or not AUDIO_AVAILABLE:
        return None, None
    try:
        audio, sr = sf.read(io.BytesIO(audio_bytes))
        if sr != target_sr:
            audio = librosa.resample(audio.astype(np.float32), orig_sr=sr, target_sr=target_sr)
            sr = target_sr
        return audio.astype(np.float32), sr
    except Exception as e:
        print(f'Error decoding audio: {e}')
        return None, None

# Decode first sample as test
if AUDIO_AVAILABLE:
    test_audio, test_sr = decode_audio(df['audio'].iloc[0])
    if test_audio is not None:
        print(f'Successfully decoded audio')
        print(f'Sample rate: {test_sr} Hz')
        print(f'Duration: {len(test_audio)/test_sr:.2f} seconds')
        print(f'Waveform shape: {test_audio.shape}')
    else:
        print('Could not decode audio — will try alternative method')
else:
    print('Audio libraries not available')

In [ ]:
# Compute durations for all samples
if AUDIO_AVAILABLE:
    durations = []
    sample_rates = []
    audio_sizes = []
    
    for i in range(len(df)):
        audio_dict = df['audio'].iloc[i]
        audio_bytes = audio_dict.get('bytes', b'')
        audio_sizes.append(len(audio_bytes))
        
        try:
            audio, sr = sf.read(io.BytesIO(audio_bytes))
            durations.append(len(audio) / sr)
            sample_rates.append(sr)
        except:
            durations.append(0)
            sample_rates.append(0)
    
    df['duration'] = durations
    df['sample_rate'] = sample_rates
    df['audio_size_kb'] = [s / 1024 for s in audio_sizes]
    
    valid = df[df['duration'] > 0]
    print('=== Audio Duration Statistics ===')
    print(f'Valid audio samples: {len(valid)} / {len(df)}')
    print(f'Mean duration: {valid["duration"].mean():.2f} seconds')
    print(f'Min duration: {valid["duration"].min():.2f} seconds')
    print(f'Max duration: {valid["duration"].max():.2f} seconds')
    print(f'Total audio: {valid["duration"].sum():.1f} seconds ({valid["duration"].sum()/60:.1f} minutes)')
    print(f'Sample rates found: {sorted(valid["sample_rate"].unique())}')
else:
    print('Skipping audio analysis — librosa/soundfile not available')

In [ ]:
# Visualize audio durations
if AUDIO_AVAILABLE and 'duration' in df.columns:
    valid = df[df['duration'] > 0]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Duration histogram
    axes[0].hist(valid['duration'], bins=30, color='#9C27B0', edgecolor='white', alpha=0.8)
    axes[0].set_xlabel('Duration (seconds)')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Distribution of Audio Durations')
    axes[0].axvline(valid['duration'].mean(), color='red', linestyle='--', 
                    label=f'Mean: {valid["duration"].mean():.1f}s')
    axes[0].legend()
    
    # Text length vs duration scatter
    axes[1].scatter(valid['text_length'], valid['duration'], alpha=0.5, color='#009688', s=30)
    axes[1].set_xlabel('Text Length (characters)')
    axes[1].set_ylabel('Audio Duration (seconds)')
    axes[1].set_title('Text Length vs Audio Duration')
    
    plt.tight_layout()
    plt.savefig('../results/figures/audio_duration_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Figure saved to results/figures/audio_duration_analysis.png')

In [ ]:
# Visualize sample waveforms and spectrograms
if AUDIO_AVAILABLE:
    fig, axes = plt.subplots(3, 2, figsize=(14, 10))
    
    for idx in range(3):
        audio, sr = decode_audio(df['audio'].iloc[idx])
        if audio is None:
            continue
        
        # Waveform
        axes[idx, 0].plot(np.linspace(0, len(audio)/sr, len(audio)), audio, 
                          color='#1565C0', linewidth=0.5)
        axes[idx, 0].set_title(f'Waveform — Sample {idx}', fontsize=10)
        axes[idx, 0].set_xlabel('Time (s)')
        axes[idx, 0].set_ylabel('Amplitude')
        
        # Mel spectrogram
        mel = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=80, n_fft=1024, hop_length=256)
        mel_db = librosa.power_to_db(mel, ref=np.max)
        img = librosa.display.specshow(mel_db, y_axis='mel', x_axis='time', 
                                       sr=sr, hop_length=256, ax=axes[idx, 1])
        axes[idx, 1].set_title(f'Mel Spectrogram — Sample {idx}', fontsize=10)
    
    plt.tight_layout()
    plt.savefig('../results/figures/sample_waveforms_spectrograms.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Figure saved to results/figures/sample_waveforms_spectrograms.png')

In [ ]:
# Listen to samples (works in Jupyter/Colab)
if AUDIO_AVAILABLE and IPYTHON_AVAILABLE:
    for i in range(min(3, len(df))):
        audio, sr = decode_audio(df['audio'].iloc[i])
        if audio is not None:
            print(f'\nSample {i}: "{df["transcription"].iloc[i]}"')
            display(Audio(audio, rate=sr))

## 5. Dataset Summary

Let's compile a comprehensive summary of our dataset.

In [ ]:
print('=' * 60)
print('     HASSANIYA ARABIC SPEECH DATASET — SUMMARY')
print('=' * 60)
print(f'  Total samples:            {len(df)}')
print(f'  Unique transcriptions:    {df["transcription"].nunique()}')
print(f'  Unique characters:        {len(char_counts)}')
print(f'  Avg text length:          {df["text_length"].mean():.1f} chars')
print(f'  Avg word count:           {df["word_count"].mean():.1f} words')
if 'duration' in df.columns:
    valid = df[df['duration'] > 0]
    print(f'  Valid audio samples:      {len(valid)}')
    print(f'  Total audio duration:     {valid["duration"].sum():.1f}s ({valid["duration"].sum()/60:.1f} min)')
    print(f'  Avg audio duration:       {valid["duration"].mean():.2f}s')
print('=' * 60)
print()
print('Key observations:')
print('  - Small but usable dataset for MVP/proof-of-concept')
print('  - Text is informal Hassaniya dialect (not MSA)')
print('  - Variable length utterances — good diversity')
print('  - Transfer learning is necessary given the dataset size')

## Conclusion

Our Hassaniya dataset contains **294 audio-text pairs** — a small but meaningful
collection for a proof-of-concept TTS system. The data shows:

1. **Text diversity**: Transcriptions range from 4 to 182 characters
2. **Dialect features**: Informal Hassaniya Arabic, distinct from Modern Standard Arabic
3. **Audio quality**: Variable durations, suitable for preprocessing pipeline development

**Next step**: Notebook 02 — Text annotation and normalization.